# Notebook 2: benchmark decomposition

Fits the univariate and external benchmark suite on the primary January 2016 to June 2024 evaluation design.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *
from shared_utils import _systematic_resample

import time
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from pytensor import scan as pt_scan

QUICK = quick_mode()
selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])
assert P == 38

d = load_data(network="geographic")
Y_train = d["Y_train"].to_numpy()
Y_full = d["Y_full"].to_numpy()
test_start = d["test_start"]
Y_hist = Y_full[:test_start]
Y_future_full = Y_full[test_start:]
test_dates_full = pd.to_datetime(d["test"].index)
countries = d["countries"]
N = d["N"]

EVAL_STEPS = min(3, len(Y_future_full)) if QUICK else len(Y_future_full)
Y_future = Y_future_full[:EVAL_STEPS]
test_dates = test_dates_full[:EVAL_STEPS]

kw = dict(NUTS_KW)
if QUICK:
    kw.update(draws=2, tune=2, chains=1, cores=1)

def scalar_ar_radius(a):
    """Return the spectral radius of a scalar AR polynomial."""
    a = np.asarray(a, dtype=float)
    return float(np.max(np.abs(np.roots(np.r_[1.0, -a]))))

def cfg(model, p=None):
    return run_config(
        p=p, stages=([0] * p if p is not None else None),
        model=model, forecast_horizon=1, seed=NUTS_KW["random_seed"],
    )

## Country AR-SV benchmarks

In [2]:
def fit_country_ar1_sv(y_i):
    """Fit one country AR(1) model with the established stochastic-volatility process."""
    y_i = np.asarray(y_i, dtype=float)
    n_time = len(y_i) - 1
    with pm.Model() as model:
        c = pm.Normal("c", 0.0, 1.0)
        a = pm.Normal("a", MINN["rw_centre"], 0.3)
        m_h = pm.Normal("m_h", -2.0, 1.0)
        phi = pm.Uniform("phi", -0.99, 0.99)
        sigma_h = pm.HalfNormal("sigma_h", 0.5)
        z = pm.Normal("h_innov", 0.0, 1.0, shape=n_time)

        def step(z_t, h_prev, m, ph, s):
            return m + ph * (h_prev - m) + s * z_t

        h0 = m_h + sigma_h * z[0] / pt.sqrt(1 - phi**2)
        h_seq, _ = pt_scan(
            fn=step, sequences=[z[1:]], outputs_info=[h0],
            non_sequences=[m_h, phi, sigma_h],
        )
        h = pm.Deterministic("h", pt.concatenate([h0[None], h_seq]))
        mean = c + a * pt.as_tensor(y_i[:-1])
        pm.Normal("obs", mu=mean, sigma=pt.exp(h / 2), observed=y_i[1:])
        idata = pm.sample(**kw)
    return idata

def fit_country_ar38_sv(y_i):
    """Fit one country AR(38) model with the established stochastic-volatility process."""
    y_i = np.asarray(y_i, dtype=float)
    X = np.column_stack([
        y_i[P-lag:len(y_i)-lag] for lag in range(1, P + 1)
    ])
    target = y_i[P:]
    n_time = len(target)

    mu_a = np.zeros(P)
    mu_a[0] = 0.8

    with pm.Model() as model:
        c = pm.Normal("c", 0.0, 1.0)
        a = pm.Normal("a", mu=mu_a, sigma=0.3, shape=P)
        m_h = pm.Normal("m_h", -2.0, 1.0)
        phi = pm.Uniform("phi", -0.99, 0.99)
        sigma_h = pm.HalfNormal("sigma_h", 0.5)
        z = pm.Normal("h_innov", 0.0, 1.0, shape=n_time)

        def step(z_t, h_prev, m, ph, s):
            return m + ph * (h_prev - m) + s * z_t

        h0 = m_h + sigma_h * z[0] / pt.sqrt(1 - phi**2)
        h_seq, _ = pt_scan(
            fn=step, sequences=[z[1:]], outputs_info=[h0],
            non_sequences=[m_h, phi, sigma_h],
        )
        h = pm.Deterministic("h", pt.concatenate([h0[None], h_seq]))
        mean = c + pt.dot(pt.as_tensor(X), a)
        pm.Normal("obs", mu=mean, sigma=pt.exp(h / 2), observed=target)
        idata = pm.sample(**kw)
    return idata

def forecast_country_ar_sv(idata, history, future, p, seed):
    """Filter one country's stochastic volatility through the evaluation block."""
    rng = np.random.default_rng(seed)
    post = idata.posterior
    c = float(post["c"].mean(("chain", "draw")).values)
    a_values = np.asarray(post["a"].mean(("chain", "draw")).values, dtype=float)
    a = np.atleast_1d(a_values)
    m_h = float(post["m_h"].mean(("chain", "draw")).values)
    phi = float(post["phi"].mean(("chain", "draw")).values)
    sigma_h = float(post["sigma_h"].mean(("chain", "draw")).values)
    h_last = float(post["h"].mean(("chain", "draw")).values[-1])

    particles = np.full(2000, h_last)
    weights = np.full(2000, 1 / 2000)
    hist = list(np.asarray(history, dtype=float))
    mean = np.zeros(len(future))
    var = np.zeros(len(future))

    for step, realised in enumerate(future):
        x = np.array([hist[-lag] for lag in range(1, p + 1)])
        mean[step] = c + a @ x

        particles = m_h + phi * (particles - m_h) + sigma_h * rng.standard_normal(len(particles))
        sigma2 = np.exp(particles)
        var[step] = float(np.average(sigma2, weights=weights))

        resid = float(realised - mean[step])
        loglik = -0.5 * (np.log(2 * np.pi * sigma2) + resid**2 / sigma2)
        loglik -= loglik.max()
        new_weights = weights * np.exp(loglik)
        total = new_weights.sum()
        if total <= 0 or not np.isfinite(total):
            raise FloatingPointError("Country SV particle weights degenerated.")
        weights = new_weights / total
        particles = particles[_systematic_resample(rng, weights)]
        weights.fill(1 / len(weights))
        hist.append(float(realised))
    return mean, var

def scalar_ar_radius(a):
    """Return the spectral radius of a scalar AR polynomial."""
    a = np.atleast_1d(np.asarray(a, dtype=float))
    return float(np.max(np.abs(np.roots(np.r_[1.0, -a]))))

def country_ar_stability(idata):
    """Summarise scalar AR stability over retained posterior draws."""
    draws = np.asarray(idata.posterior["a"].values, dtype=float)
    if draws.ndim == 2:
        draws = draws[..., None]
    draws = draws.reshape(-1, draws.shape[-1])
    radii = np.array([scalar_ar_radius(a) for a in draws])
    mean_a = draws.mean(axis=0)
    return {
        "posterior_mean_radius": scalar_ar_radius(mean_a),
        "stable_fraction": float(np.mean(radii < 1)),
        "max_radius": float(radii.max()),
    }

def run_country_ar_sv(p):
    """Fit all 23 country AR-SV models at one lag order."""
    mean = np.zeros_like(Y_future)
    var = np.zeros_like(Y_future)
    diagnostics, stability = [], []

    t0 = time.time()
    for i, country in enumerate(countries):
        idata = fit_country_ar1_sv(Y_train[:, i]) if p == 1 else fit_country_ar38_sv(Y_train[:, i])
        diagnostics.append({
            "country": country,
            **mcmc_diagnostics(idata, var_names=["c", "a", "m_h", "phi", "sigma_h", "h_innov"]),
        })
        stability.append({"country": country, **country_ar_stability(idata)})
        mean[:, i], var[:, i] = forecast_country_ar_sv(
            idata, Y_hist[:, i], Y_future[:, i], p, seed=42 + i
        )

    name = f"nb2_country_ar{p}_sv"
    save_forecasts(
        name, Y_future, mean, var, test_dates,
        config=cfg(f"country AR({p})-SV", p),
    )
    output = {
        "forecast_bundle": name,
        "score": score_forecasts(Y_future, mean, var, test_dates),
        "diagnostics": diagnostics,
        "stability": stability,
        "runtime_sec": float(time.time() - t0),
    }
    print(f"country AR({p})-SV complete in {output['runtime_sec']:.1f}s")
    return output

country_outputs = {}

In [3]:
country_outputs[1] = run_country_ar_sv(1)

/var/folders/d5/bt4mpp9n3gngv4q0rms4ylwm0000gn/T/ipykernel_1747/4198073792.py:17: DeprecationWarning: Scan return signature will change. Updates dict will not be returned, only the first argument. Pass `return_updates=False` to conform to the new API and avoid this warning
  h_seq, _ = pt_scan(
Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [c, a, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 13 seconds.


...

Multiprocess sampling (4 chains in 4 jobs)


NUTS: [c, a, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 22 seconds.


country AR(1)-SV complete in 413.0s


In [4]:
country_outputs[P] = run_country_ar_sv(P)

/var/folders/d5/bt4mpp9n3gngv4q0rms4ylwm0000gn/T/ipykernel_1747/4198073792.py:51: DeprecationWarning: Scan return signature will change. Updates dict will not be returned, only the first argument. Pass `return_updates=False` to conform to the new API and avoid this warning
  h_seq, _ = pt_scan(
Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [c, a, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 111 seconds.


...

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 69 seconds.


/var/folders/d5/bt4mpp9n3gngv4q0rms4ylwm0000gn/T/ipykernel_1747/4198073792.py:51: DeprecationWarning: Scan return signature will change. Updates dict will not be returned, only the first argument. Pass `return_updates=False` to conform to the new API and avoid this warning
  h_seq, _ = pt_scan(
Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [c, a, m_h, phi, sigma_h, h_innov]


Output()

## BVAR(1)

In [5]:
def fit_bvar(Y, lam=MINN["lambda1"]):
    """Fit a VAR(1) with a Minnesota-style prior and diagonal innovation scale."""
    Ylag, Ytar = Y[:-1], Y[1:]
    n = Y.shape[1]
    mu_A = np.eye(n)
    sd_A = np.full((n, n), lam * MINN["lambda2"])
    np.fill_diagonal(sd_A, lam)

    with pm.Model() as model:
        A = pm.Normal("A", mu=mu_A, sigma=sd_A, shape=(n, n))
        sig = pm.HalfNormal("sig", 1.0, shape=n)
        pm.Normal("obs", mu=pt.dot(pt.as_tensor(Ylag), A.T), sigma=sig, observed=Ytar)
        idata = pm.sample(**kw)

    A_hat = idata.posterior["A"].mean(("chain", "draw")).values
    sig_hat = idata.posterior["sig"].mean(("chain", "draw")).values
    return idata, A_hat, sig_hat

def bvar_forecast(A, sig):
    """One-step VAR forecasts on realised evaluation history."""
    mean = np.zeros_like(Y_future)
    var = np.tile(sig**2, (len(Y_future), 1))
    last_value = Y_hist[-1]
    for step, realised in enumerate(Y_future):
        mean[step] = A @ last_value
        last_value = realised
    return mean, var

idata_bvar, A_hat, sig_hat = fit_bvar(Y_train)
mean_bvar, var_bvar = bvar_forecast(A_hat, sig_hat)
save_forecasts(
    "nb2_bvar1", Y_future, mean_bvar, var_bvar, test_dates,
    config=run_config(p=1, stages=None, model="BVAR(1)", forecast_horizon=1,
                      seed=NUTS_KW["random_seed"]),
)
bvar_diag = mcmc_diagnostics(idata_bvar, var_names=["A", "sig"])

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [A, sig]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 21 seconds.


## GARCH benchmarks

In [6]:
from arch import arch_model

def garch_forecast(ar_lags):
    """Run rolling AR-GARCH(1,1) forecasts with the established scaling convention."""
    mean = np.full_like(Y_future, np.nan)
    var = np.full_like(Y_future, np.nan)
    convergence_failures = 0
    fit_exceptions = 0
    failed_fit_attempts = 0
    convergence_flags = []

    for i in range(N):
        series = np.concatenate([Y_hist[:, i], Y_future[:, i]]) * 10.0
        for step in range(len(Y_future)):
            end = len(Y_hist) + step
            failed = False
            try:
                am = arch_model(
                    series[:end], mean="AR", lags=ar_lags,
                    vol="Garch", p=1, q=1, rescale=False,
                )
                res = am.fit(disp="off", show_warning=False)
                flag = int(getattr(res, "convergence_flag", 0))
                convergence_flags.append({
                    "country": countries[i], "date": test_dates[step], "flag": flag,
                })
                if flag != 0:
                    convergence_failures += 1
                    failed = True
                fc = res.forecast(horizon=1, reindex=False)
                mean[step, i] = fc.mean.values[-1, 0] / 10.0
                var[step, i] = fc.variance.values[-1, 0] / 100.0
            except Exception:
                fit_exceptions += 1
                failed = True
            if failed:
                failed_fit_attempts += 1

    valid = np.isfinite(mean) & np.isfinite(var) & (var > 0)
    diagnostics = {
        "fit_attempts": int(N * len(Y_future)),
        "optimizer_failures": int(convergence_failures),
        "fit_exceptions": int(fit_exceptions),
        "failed_fit_attempts": int(failed_fit_attempts),
        "failed_fit_proportion": float(failed_fit_attempts / (N * len(Y_future))),
        "invalid_forecast_cells": int((~valid).sum()),
        "convergence_flags": convergence_flags,
    }
    return mean, var, diagnostics

garch_outputs = {}
for p in [1, P]:
    t0 = time.time()
    mean, var, diag = garch_forecast(p)
    complete = bool(np.all(np.isfinite(mean)) and np.all(np.isfinite(var)) and np.all(var > 0))
    bundle = f"nb2_ar{p}_garch"
    if complete:
        save_forecasts(
            bundle, Y_future, mean, var, test_dates,
            config=run_config(p=p, stages=None, model=f"AR({p})-GARCH(1,1)", forecast_horizon=1),
        )
        score = score_forecasts(Y_future, mean, var, test_dates)
    else:
        score = None
    garch_outputs[p] = {
        "forecast_bundle": bundle if complete else None,
        "score": score,
        "diagnostics": diag,
        "runtime_sec": float(time.time() - t0),
    }
    print(f"AR({p})-GARCH: {diag}")

AR(1)-GARCH: {'fit_attempts': 2346, 'optimizer_failures': 0, 'fit_exceptions': 0, 'failed_fit_attempts': 0, 'failed_fit_proportion': 0.0, 'invalid_forecast_cells': 0, 'convergence_flags': [{'country': 'PRT', 'date': Timestamp('2016-01-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-02-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-03-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-04-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-05-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-06-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-07-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-08-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-09-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-10-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-11-01 00:00:00'), 'flag': 0}, {'countr

AR(38)-GARCH: {'fit_attempts': 2346, 'optimizer_failures': 2, 'fit_exceptions': 0, 'failed_fit_attempts': 2, 'failed_fit_proportion': 0.0008525149190110827, 'invalid_forecast_cells': 0, 'convergence_flags': [{'country': 'PRT', 'date': Timestamp('2016-01-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-02-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-03-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-04-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-05-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-06-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-07-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-08-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-09-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-10-01 00:00:00'), 'flag': 0}, {'country': 'PRT', 'date': Timestamp('2016-11-01 00:00:00'), '

## Random walk

In [7]:
mean_rw = np.zeros_like(Y_future)
last_value = Y_hist[-1]
for step, realised in enumerate(Y_future):
    mean_rw[step] = last_value
    last_value = realised

sig_rw = float(np.std(np.diff(Y_hist, axis=0)))
var_rw = np.full_like(Y_future, sig_rw**2)
save_forecasts(
    "nb2_random_walk", Y_future, mean_rw, var_rw, test_dates,
    config=run_config(model="random walk", forecast_horizon=1),
)
score_rw = score_forecasts(Y_future, mean_rw, var_rw, test_dates)

## UCSV

In [8]:
UCSV_GAMMA = 0.2
UCSV_LOGVAR_SD = np.sqrt(UCSV_GAMMA)

def fit_ucsv_country(y):
    """Fit one Stock-Watson-form UCSV model and return terminal posterior draws."""
    n = len(y)
    dy = np.diff(y)
    g0 = float(np.var(dy))
    g1 = float(np.cov(dy[:-1], dy[1:])[0, 1])
    s2_eps = max(-g1, 1e-6)
    s2_eta = max(g0 + 2 * g1, 1e-6)
    lv_obs, lv_trend = float(np.log(s2_eps)), float(np.log(s2_eta))

    with pm.Model() as model:
        h_e0 = pm.Normal("h_e0", lv_obs, 1.0)
        h_n0 = pm.Normal("h_n0", lv_trend, 1.0)
        z_he = pm.Normal("z_he", 0.0, 1.0, shape=n)
        z_hn = pm.Normal("z_hn", 0.0, 1.0, shape=n)
        h_e = pm.Deterministic(
            "h_e", h_e0 + pt.cumsum(UCSV_LOGVAR_SD * z_he) - UCSV_LOGVAR_SD * z_he[0]
        )
        h_n = pm.Deterministic(
            "h_n", h_n0 + pt.cumsum(UCSV_LOGVAR_SD * z_hn) - UCSV_LOGVAR_SD * z_hn[0]
        )

        tau0 = pm.Normal("tau0", float(y[0]), 1.0)
        z_eta = pm.Normal("z_eta", 0.0, 1.0, shape=n)
        tau = pm.Deterministic("tau", tau0 + pt.cumsum(pt.exp(h_n / 2) * z_eta))
        pm.Normal("obs", mu=tau, sigma=pt.exp(h_e / 2), observed=y)
        idata = pm.sample(**kw)

    post = idata.posterior
    tau_end = post["tau"].values[..., -1].ravel()
    draws = {
        "tau": tau_end,
        "h_e": post["h_e"].values[..., -1].ravel(),
        "h_n": post["h_n"].values[..., -1].ravel(),
        "g_e": np.full(tau_end.shape, UCSV_LOGVAR_SD),
        "g_n": np.full(tau_end.shape, UCSV_LOGVAR_SD),
    }
    diag = mcmc_diagnostics(
        idata, var_names=["tau0", "h_e0", "h_n0", "z_he", "z_hn", "z_eta"]
    )
    return draws, diag

def ucsv_filter_country(draws, future, seed):
    """Filter one UCSV posterior through an evaluation block."""
    rng = np.random.default_rng(seed)
    M = 2000
    idx = rng.integers(0, len(draws["tau"]), size=M)
    tau = draws["tau"][idx].copy()
    h_e = draws["h_e"][idx].copy()
    h_n = draws["h_n"][idx].copy()
    g_e = draws["g_e"][idx].copy()
    g_n = draws["g_n"][idx].copy()
    w = np.full(M, 1 / M)

    mean, var = np.zeros(len(future)), np.zeros(len(future))
    clip_counts = {"h_e": 0, "h_n": 0}

    for step, realised in enumerate(future):
        h_e_proposed = h_e + g_e * rng.standard_normal(M)
        h_n_proposed = h_n + g_n * rng.standard_normal(M)

        clip_counts["h_e"] += int(
            np.count_nonzero((h_e_proposed < -20.0) | (h_e_proposed > 10.0))
        )
        clip_counts["h_n"] += int(
            np.count_nonzero((h_n_proposed < -20.0) | (h_n_proposed > 10.0))
        )

        h_e = np.clip(h_e_proposed, -20.0, 10.0)
        h_n = np.clip(h_n_proposed, -20.0, 10.0)

        tau = tau + np.exp(h_n / 2) * rng.standard_normal(M)
        v_obs = np.exp(h_e)

        mu = float(np.sum(w * tau))
        mean[step] = mu
        var[step] = max(
            float(np.sum(w * (v_obs + tau**2)) - mu**2),
            1e-12,
        )

        sd = np.sqrt(np.maximum(v_obs, 1e-12))
        logw = (
            np.log(np.maximum(w, 1e-300))
            - 0.5 * ((realised - tau) / sd)**2
            - np.log(sd)
        )
        logw -= logw.max()
        w = np.exp(logw)
        w /= w.sum()

        take = _systematic_resample(rng, w)
        tau, h_e, h_n = tau[take], h_e[take], h_n[take]
        g_e, g_n = g_e[take], g_n[take]
        w.fill(1 / M)

    return mean, var, clip_counts

mean_ucsv = np.zeros_like(Y_future)
var_ucsv = np.zeros_like(Y_future)
ucsv_diagnostics = []
ucsv_clip_counts = []

t0 = time.time()
for i, country in enumerate(countries):
    draws, diag = fit_ucsv_country(Y_train[:, i])
    ucsv_diagnostics.append({"country": country, **diag})

    mean_i, var_i, clips = ucsv_filter_country(
        draws, Y_future[:, i], 42 + i
    )
    mean_ucsv[:, i] = mean_i
    var_ucsv[:, i] = var_i
    ucsv_clip_counts.append({"country": country, **clips})

ucsv_total_clips = {
    state: int(sum(row[state] for row in ucsv_clip_counts))
    for state in ["h_e", "h_n"]
}

save_forecasts(
    "nb2_ucsv", Y_future, mean_ucsv, var_ucsv, test_dates,
    config=run_config(model="UCSV", forecast_horizon=1, seed=NUTS_KW["random_seed"],
                      log_variance_innovation_variance=UCSV_GAMMA),
)
score_ucsv = score_forecasts(Y_future, mean_ucsv, var_ucsv, test_dates)
ucsv_runtime = float(time.time() - t0)
print(
    f"UCSV complete in {ucsv_runtime:.1f}s; "
    f"divergences={sum(x['divergences'] for x in ucsv_diagnostics)}; "
    f"clips={ucsv_total_clips}"
)

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [h_e0, h_n0, z_he, z_hn, tau0, z_eta]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 78 seconds.


...

The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [h_e0, h_n0, z_he, z_hn, tau0, z_eta]


Output()

## Benchmark summary

In [9]:
benchmark_bundles = {
    "country_ar1_sv": country_outputs[1]["forecast_bundle"],
    "country_ar38_sv": country_outputs[P]["forecast_bundle"],
    "bvar1": "nb2_bvar1",
    "ar1_garch": garch_outputs[1]["forecast_bundle"],
    "ar38_garch": garch_outputs[P]["forecast_bundle"],
    "random_walk": "nb2_random_walk",
    "ucsv": "nb2_ucsv",
}

score_rows = [
    {"model": "country_ar1_sv", **country_outputs[1]["score"]},
    {"model": "country_ar38_sv", **country_outputs[P]["score"]},
    {"model": "bvar1", **score_forecasts(Y_future, mean_bvar, var_bvar, test_dates)},
    {"model": "random_walk", **score_rw},
    {"model": "ucsv", **score_ucsv},
]
for p in [1, P]:
    if garch_outputs[p]["score"] is not None:
        score_rows.append({"model": f"ar{p}_garch", **garch_outputs[p]["score"]})

benchmark_table = pd.DataFrame(score_rows)
print(benchmark_table[["model", "CRPS", "RMSE", "MAE", "Coverage_95", "Width"]]
      .round(6).to_string(index=False))

save_result("nb2_benchmarks", {
    "country_ar_sv": {str(k): v for k, v in country_outputs.items()},
    "bvar_diagnostics": bvar_diag,
    "garch": {str(k): v for k, v in garch_outputs.items()},
    "random_walk": {"score": score_rw, "pooled_first_difference_sd": sig_rw},
    "ucsv": {
        "score": score_ucsv,
        "diagnostics": ucsv_diagnostics,
        "runtime_sec": ucsv_runtime,
        "gamma": UCSV_GAMMA,
        "log_variance_innovation_sd": float(UCSV_LOGVAR_SD),
        "clip_counts": ucsv_clip_counts,
        "total_clip_counts": ucsv_total_clips,
    },
    "table": benchmark_table.to_dict(orient="records"),
    "forecast_bundles": benchmark_bundles,
    "config": run_config(p=P, stages=[0] * P, purpose="benchmark_suite"),
    "quick": QUICK,
})
print("saved nb2_benchmarks")

          model     CRPS     RMSE      MAE  Coverage_95    Width
 country_ar1_sv 0.279197 0.559074 0.377299     0.890878 1.575204
country_ar38_sv 0.236950 0.460455 0.318496     0.858909 1.221689
          bvar1 0.328151 0.626716 0.426161     0.815857 1.525171
    random_walk 0.284639 0.557456 0.374472     0.922847 1.844912
           ucsv 0.284305 0.568626 0.382811     0.949275 2.267194
      ar1_garch 0.275302 0.559377 0.377140     0.907502 1.712350
     ar38_garch 0.231043 0.452323 0.312733     0.875959 1.320820
saved nb2_benchmarks
